In [12]:
import pandas as pd
import numpy as np
import yaml
import re
import os
from itertools import permutations
DB_FOLDER = r"dataset\database_v3"
GRAPH_FOLDER = os.path.join(DB_FOLDER, "Graph_Size")

In [13]:
node_df = pd.read_csv(os.path.join(GRAPH_FOLDER, "node_id.csv"))

node_lookup = (
    node_df[["project_id", "node_id", "size_bucket"]]
    .drop_duplicates()
    .copy()
)

def expand_project_edges_to_node_edges(project_edges, node_lookup):
    """Convert project-level edges to node-level edges with matching size_bucket."""
    if project_edges.empty:
        return pd.DataFrame(columns=["node_id", "neig_id"])

    source_nodes = node_lookup.copy()
    target_nodes = node_lookup.rename(
        columns={"project_id": "neig_project_id", "node_id": "neig_id"}
    )

    node_edges = (
        project_edges
        .merge(source_nodes, on="project_id", how="inner")
        .merge(target_nodes, on=["neig_project_id", "size_bucket"], how="inner")
    )

    return (
        node_edges.loc[node_edges["node_id"] != node_edges["neig_id"], ["node_id", "neig_id"]]
        .drop_duplicates()
        .sort_values(["node_id", "neig_id"])
        .reset_index(drop=True)
    )

node_df

,node_id,project_id,Project Name,size_bucket,node_key,count
0,0,0,# 1 LOFT,SZ1,0 | SZ1,229
1,1,0,# 1 LOFT,SZ2,0 | SZ2,13
2,2,0,# 1 LOFT,SZ3,0 | SZ3,12
3,3,0,# 1 LOFT,SZ4,0 | SZ4,16
4,4,1,# 1 SUITES,SZ1,1 | SZ1,240
...,...,...,...,...,...,...
7953,7953,2495,ZENITH,SZ4,2495 | SZ4,27
7954,7954,2495,ZENITH,SZ5,2495 | SZ5,9
7955,7955,2496,ZYANYA,SZ2,2496 | SZ2,8
7956,7956,2496,ZYANYA,SZ3,2496 | SZ3,2


In [14]:
# edges_list = []

# # --------------------------------------------------
# # 1. build edges within same project
# # --------------------------------------------------
# for project_id, group in node_df.groupby('project_id'):
#     node_ids = group['node_id'].tolist()
    
#     # create all ordered pairs (i, j) where i != j
#     for i, j in permutations(node_ids, 2):
#         edges_list.append((i, j))

# # --------------------------------------------------
# # 2. convert to dataframe
# # --------------------------------------------------
# edges_df = pd.DataFrame(edges_list, columns=['node_id', 'neig_id'])

# # --------------------------------------------------
# # 3. save
# # --------------------------------------------------
# edge_dir = os.path.join(DB_FOLDER, "edges")
# os.makedirs(edge_dir, exist_ok=True)

# edges_df.to_csv(os.path.join(edge_dir, "same_project.csv"), index=False)

In [15]:
threshold = 250  # meters

project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))

# --------------------------------------------------
# 1. keep unique project coordinates
# --------------------------------------------------
proj_coord = (
    project_df[['project_id', 'latitude', 'longitude']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

proj_coord['latitude'] = pd.to_numeric(proj_coord['latitude'], errors='coerce')
proj_coord['longitude'] = pd.to_numeric(proj_coord['longitude'], errors='coerce')
proj_coord = proj_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 2. haversine distance (meters)
# --------------------------------------------------
R = 6371000.0

lat = np.radians(proj_coord['latitude'].to_numpy())
lon = np.radians(proj_coord['longitude'].to_numpy())

dlat = lat[:, None] - lat[None, :]
dlon = lon[:, None] - lon[None, :]

a = np.sin(dlat / 2)**2 + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin(dlon / 2)**2
c = 2 * np.arcsin(np.sqrt(a))
dist_mat = R * c

# --------------------------------------------------
# 3. build edges (<= threshold, exclude self)
# --------------------------------------------------
mask = (dist_mat <= threshold) & (dist_mat > 0)

src_idx, dst_idx = np.where(mask)

project_edges_df = pd.DataFrame({
    'project_id': proj_coord.iloc[src_idx]['project_id'].to_numpy(),
    'neig_project_id': proj_coord.iloc[dst_idx]['project_id'].to_numpy(),
})

edges_df = expand_project_edges_to_node_edges(project_edges_df, node_lookup)

# --------------------------------------------------
# 4. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df.to_csv(os.path.join(edge_dir, f"dist_{threshold}.csv"), index=False)

print(f"Saved distance node edges with matching size_bucket: {len(edges_df):,}")
display(edges_df.head())

Saved distance node edges with matching size_bucket: 59,770


,node_id,neig_id
0,0,4
1,0,73
2,0,410
3,0,881
4,0,1046


In [16]:
radius = 500  # meters

project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))
mrt_df = pd.read_csv(os.path.join(DB_FOLDER, "Rail_Transport.csv"))

# --------------------------------------------------
# 1. keep unique project coordinates
# --------------------------------------------------
proj_coord = (
    project_df[['project_id', 'latitude', 'longitude']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

proj_coord['latitude'] = pd.to_numeric(proj_coord['latitude'], errors='coerce')
proj_coord['longitude'] = pd.to_numeric(proj_coord['longitude'], errors='coerce')
proj_coord = proj_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 2. prepare MRT coordinates
#    adjust these column names if your file uses different names
# --------------------------------------------------
mrt_coord = mrt_df[['latitude', 'longitude']].copy()
mrt_coord['latitude'] = pd.to_numeric(mrt_coord['latitude'], errors='coerce')
mrt_coord['longitude'] = pd.to_numeric(mrt_coord['longitude'], errors='coerce')
mrt_coord = mrt_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 3. haversine distance: project -> MRT
# --------------------------------------------------
R = 6371000.0  # Earth radius in meters

proj_lat = np.radians(proj_coord['latitude'].to_numpy())
proj_lon = np.radians(proj_coord['longitude'].to_numpy())

mrt_lat = np.radians(mrt_coord['latitude'].to_numpy())
mrt_lon = np.radians(mrt_coord['longitude'].to_numpy())

dlat = proj_lat[:, None] - mrt_lat[None, :]
dlon = proj_lon[:, None] - mrt_lon[None, :]

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(proj_lat)[:, None] * np.cos(mrt_lat)[None, :] * np.sin(dlon / 2.0) ** 2
)
c = 2 * np.arcsin(np.sqrt(a))
proj_mrt_dist = R * c  # shape: [num_projects, num_mrt]

# --------------------------------------------------
# 4. membership matrix:
#    within radius of each MRT circle
# --------------------------------------------------
within_circle = proj_mrt_dist <= radius  # bool matrix [P, M]

# --------------------------------------------------
# 5. if two projects share at least one MRT circle, connect them
# --------------------------------------------------
# project-project shared-circle count
shared_circle = within_circle @ within_circle.T  # bool works like 0/1 in matrix multiply

# edge if share >=1 circle, exclude self
edge_mask = (shared_circle > 0)
np.fill_diagonal(edge_mask, False)

src_idx, dst_idx = np.where(edge_mask)

project_edges_df = pd.DataFrame({
    'project_id': proj_coord.iloc[src_idx]['project_id'].to_numpy(),
    'neig_project_id': proj_coord.iloc[dst_idx]['project_id'].to_numpy(),
})

project_edges_df = (
    project_edges_df
    .drop_duplicates()
    .sort_values(['project_id', 'neig_project_id'])
    .reset_index(drop=True)
)

edges_df = expand_project_edges_to_node_edges(project_edges_df, node_lookup)

# --------------------------------------------------
# 6. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df.to_csv(os.path.join(edge_dir, f"mrt_cir_{radius}.csv"), index=False)

print(f"Saved MRT-circle node edges with matching size_bucket: {len(edges_df):,}")
display(edges_df.head())


Saved MRT-circle node edges with matching size_bucket: 53,912


,node_id,neig_id
0,0,4
1,0,410
2,0,1035
3,0,1046
4,0,1047


In [21]:
eps = 1  # meters

project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))
mrt_df = pd.read_csv(os.path.join(DB_FOLDER, "Rail_Transport.csv"))

# --------------------------------------------------
# 1. keep unique project coordinates
# --------------------------------------------------
proj_coord = (
    project_df[['project_id', 'latitude', 'longitude']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

proj_coord['latitude'] = pd.to_numeric(proj_coord['latitude'], errors='coerce')
proj_coord['longitude'] = pd.to_numeric(proj_coord['longitude'], errors='coerce')
proj_coord = proj_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 2. prepare MRT coordinates
# --------------------------------------------------
mrt_coord = mrt_df[['latitude', 'longitude']].copy()
mrt_coord['latitude'] = pd.to_numeric(mrt_coord['latitude'], errors='coerce')
mrt_coord['longitude'] = pd.to_numeric(mrt_coord['longitude'], errors='coerce')
mrt_coord = mrt_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 3. haversine distance: project -> MRT
# --------------------------------------------------
R = 6371000.0  # Earth radius in meters

proj_lat = np.radians(proj_coord['latitude'].to_numpy())
proj_lon = np.radians(proj_coord['longitude'].to_numpy())

mrt_lat = np.radians(mrt_coord['latitude'].to_numpy())
mrt_lon = np.radians(mrt_coord['longitude'].to_numpy())

dlat = proj_lat[:, None] - mrt_lat[None, :]
dlon = proj_lon[:, None] - mrt_lon[None, :]

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(proj_lat)[:, None] * np.cos(mrt_lat)[None, :] * np.sin(dlon / 2.0) ** 2
)
c = 2 * np.arcsin(np.sqrt(a))
proj_mrt_dist = R * c  # shape: [num_projects, num_mrt]

# --------------------------------------------------
# 4. nearest MRT distance for each project
# --------------------------------------------------
nearest_mrt_dist = proj_mrt_dist.min(axis=1)

# --------------------------------------------------
# 5. if two projects have similar nearest-MRT distance, connect them
# --------------------------------------------------
dist_diff = np.abs(nearest_mrt_dist[:, None] - nearest_mrt_dist[None, :])
edge_mask = dist_diff < eps
np.fill_diagonal(edge_mask, False)

src_idx, dst_idx = np.where(edge_mask)

project_edges_df = pd.DataFrame({
    'project_id': proj_coord.iloc[src_idx]['project_id'].to_numpy(),
    'neig_project_id': proj_coord.iloc[dst_idx]['project_id'].to_numpy(),
})

project_edges_df = (
    project_edges_df
    .drop_duplicates()
    .sort_values(['project_id', 'neig_project_id'])
    .reset_index(drop=True)
)

edges_df = expand_project_edges_to_node_edges(project_edges_df, node_lookup)

# --------------------------------------------------
# 6. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df.to_csv(os.path.join(edge_dir, f"mrt_nearest_dist_eps_{eps}.csv"), index=False)

print(f"Saved nearest-MRT-distance node edges with matching size_bucket: {len(edges_df):,}")
display(edges_df.head())


Saved nearest-MRT-distance node edges with matching size_bucket: 22,244


,node_id,neig_id
0,0,3953
1,1,3954
2,1,5307
3,1,6173
4,2,1370


In [18]:
project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))

# --------------------------------------------------
# 1. keep projects with non-empty Condo_Age_2026
# --------------------------------------------------
project_age = (
    project_df[['project_id', 'Condo_Age_2026']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

age_text = project_age['Condo_Age_2026'].astype('string').str.strip()
project_age = project_age[age_text.notna() & (age_text != '')].copy()
project_age['Condo_Age_2026'] = age_text[project_age.index]

# --------------------------------------------------
# 2. connect projects with the same Condo_Age_2026
# --------------------------------------------------
edges_list = []

for _, group in project_age.groupby('Condo_Age_2026'):
    project_ids = group['project_id'].drop_duplicates().to_numpy()
    if len(project_ids) < 2:
        continue

    for project_id, neig_project_id in permutations(project_ids, 2):
        edges_list.append((project_id, neig_project_id))

project_edges_df = pd.DataFrame(edges_list, columns=['project_id', 'neig_project_id'])

project_edges_df = (
    project_edges_df
    .drop_duplicates()
    .sort_values(['project_id', 'neig_project_id'])
    .reset_index(drop=True)
)

edges_df = expand_project_edges_to_node_edges(project_edges_df, node_lookup)

# --------------------------------------------------
# 3. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df.to_csv(os.path.join(edge_dir, "same_condo_age_2026.csv"), index=False)

print(f"Saved same-condo-age node edges with matching size_bucket: {len(edges_df):,}")
display(edges_df.head())


Saved same-condo-age node edges with matching size_bucket: 94,336


,node_id,neig_id
0,10,230
1,10,310
2,10,1179
3,10,1350
4,10,1442


In [19]:
# Edge file 1: connect all size-bucket nodes that belong to the same project.
# Example: project_id=0 has SZ2 and SZ4, so its two node_ids are linked both ways.
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_list = []

for project_id, group in node_df.groupby("project_id"):
    node_ids = group["node_id"].drop_duplicates().tolist()
    if len(node_ids) < 2:
        continue

    for node_id, neig_id in permutations(node_ids, 2):
        edges_list.append((node_id, neig_id))

same_project_node_edges = pd.DataFrame(
    edges_list,
    columns=["node_id", "neig_id"],
)

same_project_node_edges = (
    same_project_node_edges
    .drop_duplicates()
    .sort_values(["node_id", "neig_id"])
    .reset_index(drop=True)
)

same_project_node_edges.to_csv(
    os.path.join(edge_dir, "same_project_node_id.csv"),
    index=False,
)

print(f"Saved same-project node edges: {len(same_project_node_edges):,}")
display(same_project_node_edges.head())

Saved same-project node edges: 21,666


,node_id,neig_id
0,0,1
1,0,2
2,0,3
3,1,0
4,1,2


In [20]:
# # Edge file 2: connect all nodes that have the same size bucket.
# # Example: all SZ2 nodes are linked to all other SZ2 nodes, regardless of project.
# edge_dir = os.path.join(GRAPH_FOLDER, "edges")
# os.makedirs(edge_dir, exist_ok=True)

# edges_list = []

# for size_bucket, group in node_df.groupby("size_bucket"):
#     node_ids = group["node_id"].drop_duplicates().tolist()
#     if len(node_ids) < 2:
#         continue

#     for node_id, neig_id in permutations(node_ids, 2):
#         # edges_list.append((node_id, neig_id, size_bucket))
#         edges_list.append((node_id, neig_id))

# same_size_bucket_node_edges = pd.DataFrame(
#     edges_list,
#     # columns=["node_id", "neig_id", "size_bucket"],
#     columns=["node_id", "neig_id"],
# )

# same_size_bucket_node_edges = (
#     same_size_bucket_node_edges
#     .drop_duplicates()
#     .sort_values(["node_id", "neig_id"])
#     .reset_index(drop=True)
# )

# same_size_bucket_node_edges.to_csv(
#     os.path.join(edge_dir, "same_size_bucket_node_id.csv"),
#     index=False,
# )

# print(f"Saved same-size-bucket node edges: {len(same_size_bucket_node_edges):,}")
# display(same_size_bucket_node_edges.head())